In [17]:
import os
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

# Function to extract 24-dimensional color histogram features from a single patch
def extract_patch_histogram(patch, bins=(8, 8, 8)):
    # Extract 8-bin color histograms for R, G, B channels
    hist_r = cv2.calcHist([patch], [0], None, [bins[0]], [0, 256]).flatten()  # Red channel
    hist_g = cv2.calcHist([patch], [1], None, [bins[1]], [0, 256]).flatten()  # Green channel
    hist_b = cv2.calcHist([patch], [2], None, [bins[2]], [0, 256]).flatten()  # Blue channel
    
    # Normalize histograms
    hist_r /= np.sum(hist_r)
    hist_g /= np.sum(hist_g)
    hist_b /= np.sum(hist_b)
    
    # Concatenate R, G, B histograms into a 24-dimensional vector
    feature_vector = np.concatenate([hist_r, hist_g, hist_b])
    
    return feature_vector

# Function to divide image into 32x32 patches and extract features from each patch
def extract_image_features(image, patch_size=(32, 32), bins=(8, 8, 8)):
    img_h, img_w, _ = image.shape
    all_patches_features = []
    
    patch_counter = 0  # Counter to keep track of the number of patches
    for i in range(0, img_h, patch_size[0]):
        for j in range(0, img_w, patch_size[1]):
            # Extract a 32x32 patch
            patch = image[i:i+patch_size[0], j:j+patch_size[1]]
            if patch.shape[0] == patch_size[0] and patch.shape[1] == patch_size[1]:
                # Extract 24-dimensional feature vector for the patch
                patch_features = extract_patch_histogram(patch, bins)
                all_patches_features.append(patch_features)
                patch_counter += 1  # Increment patch counter
    
    # Stack all patch feature vectors for the image
    return np.array(all_patches_features)

# Function to save feature vectors to a file
def save_feature_vectors(feature_vectors, save_path):
    np.save(save_path, feature_vectors)  # Save as .npy file for efficient storage

# Function to process all images in a folder and extract features
def process_images_from_folder(folder_path, output_folder):
    for class_folder in os.listdir(folder_path):
        class_folder_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_folder_path):
            for image_name in os.listdir(class_folder_path):
                image_path = os.path.join(class_folder_path, image_name)
                # Load image
                image = cv2.imread(image_path)
                if image is not None:
                    # Extract features
                    features = extract_image_features(image)
                    
                    # Save features
                    output_class_folder = os.path.join(output_folder, class_folder)
                    if not os.path.exists(output_class_folder):
                        os.makedirs(output_class_folder)
                    save_feature_vectors(features, os.path.join(output_class_folder, f"{image_name.split('.')[0]}_features.npy"))
                    print(f"Processed and saved features for {image_name}")

# Example usage for training and test set
def main():
    train_folder = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\train'  # Folder containing the train images (botanical_garden, bus_interior, elevator_shaft)
    test_folder = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\test'    # Folder containing the test images (botanical_garden, bus_interior, elevator_shaft)
    
    output_train_features = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\train\output_train1_feature'  # Where to save the extracted train features
    output_test_features = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\test\output_test1_feature'    # Where to save the extracted test features
    
    print("Processing training images...")
    process_images_from_folder(train_folder, output_train_features)
    
    print("Processing test images...")
    process_images_from_folder(test_folder, output_test_features)

if __name__ == '__main__':
    main()


Processing training images...
Processed and saved features for sun_aagvwgntclshghai.jpg
Processed and saved features for sun_aamlzecjkxlnoedl.jpg
Processed and saved features for sun_aarrlvxxrjxftsuu.jpg
Processed and saved features for sun_abjxfjcrqxjhbkhi.jpg
Processed and saved features for sun_abybmfbnmlztrjpm.jpg
Processed and saved features for sun_acrbvetgiwtbvpid.jpg
Processed and saved features for sun_acrvwdwcnqgbvcbi.jpg
Processed and saved features for sun_adlsavfxcjkctdfu.jpg
Processed and saved features for sun_aeylitslkauqmnqw.jpg
Processed and saved features for sun_afqgbxrmirjsvmid.jpg
Processed and saved features for sun_afxnftrhigdckaaa.jpg
Processed and saved features for sun_aghkmbqrsfmpshxw.jpg
Processed and saved features for sun_aglxrcfjrogfzdhm.jpg
Processed and saved features for sun_ahtzdwjqixilslrl.jpg
Processed and saved features for sun_airytsbhvjoeuqnk.jpg
Processed and saved features for sun_aiuhwofaknavsqzr.jpg
Processed and saved features for sun_akbyd

In [ ]:
import os
import numpy as np

# Function to load feature vectors from a folder
def load_features_from_folder(folder_path):
    all_features = []
    for feature_file in os.listdir(folder_path):
        if feature_file.endswith(".npy"):  # Ensure we're reading only .npy files
            feature_path = os.path.join(folder_path, feature_file)
            features = np.load(feature_path)  # Load the features
            all_features.append(features)  # Append the features to the list

    # Stack all features vertically to create a single feature array
    return np.vstack(all_features), None  # None placeholder for labels if needed

# Step 1: K-means clustering from scratch

def initialize_centroids(X, k):
    """ Randomly initialize k centroids from the data points X. """
    indices = np.random.choice(len(X), k, replace=False)
    return X[indices]

def assign_clusters(X, centroids):
    """ Assign each data point in X to the nearest centroid. """
    clusters = []
    for x in X:
        distances = [np.linalg.norm(x - centroid) for centroid in centroids]
        clusters.append(np.argmin(distances))
    return np.array(clusters)

def update_centroids(X, clusters, k):
    """ Update centroids by calculating the mean of all points assigned to each centroid. """
    new_centroids = []
    for i in range(k):
        cluster_points = X[clusters == i]
        if len(cluster_points) > 0:
            new_centroids.append(np.mean(cluster_points, axis=0))
        else:
            # Handle empty cluster by reinitializing randomly
            new_centroids.append(np.random.rand(X.shape[1]))  
    return np.array(new_centroids)

def kmeans(X, k, max_iters=100):
    """ K-means clustering algorithm. """
    centroids = initialize_centroids(X, k)
    for _ in range(max_iters):
        clusters = assign_clusters(X, centroids)
        new_centroids = update_centroids(X, clusters, k)
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
    return centroids, clusters

# Step 2: Gaussian Mixture Model (GMM) using Expectation-Maximization (EM) from scratch

def gaussian_pdf(x, mean, cov):
    """ Multivariate Gaussian Probability Density Function. """
    n = len(x)
    cov_det = np.linalg.det(cov)
    if cov_det == 0:
        cov_det = 1e-6  # Handle numerical instability
    diff = x - mean
    cov_inv = np.linalg.inv(cov + np.eye(len(cov)) * 1e-6)  # Regularize covariance matrix
    exponent = -0.5 * np.dot(diff.T, np.dot(cov_inv, diff))
    return np.exp(exponent) / np.sqrt((2 * np.pi)**n * cov_det)

def expectation_step(X, means, covariances, weights):
    """ E-step: Calculate the responsibilities of each Gaussian component. """
    n, k = X.shape[0], len(means)
    responsibilities = np.zeros((n, k))
    for i in range(k):
        for j in range(n):
            responsibilities[j, i] = weights[i] * gaussian_pdf(X[j], means[i], covariances[i])
    
    # To avoid division by zero, add a small constant to the denominator
    responsibilities_sum = responsibilities.sum(axis=1, keepdims=True)
    responsibilities_sum[responsibilities_sum == 0] = 1e-6  # Avoid divide by zero
    responsibilities /= responsibilities_sum
    return responsibilities

def maximization_step(X, responsibilities):
    """ M-step: Update the means, covariances, and weights based on responsibilities. """
    n, d = X.shape
    k = responsibilities.shape[1]
    weights = responsibilities.sum(axis=0) / n
    means = np.dot(responsibilities.T, X) / responsibilities.sum(axis=0)[:, None]
    covariances = []
    for i in range(k):
        diff = X - means[i]
        covariances.append(np.dot((responsibilities[:, i] * diff.T), diff) / responsibilities[:, i].sum())
    return means, covariances, weights

def gmm_em(X, k, max_iters=100, tol=1e-6):
    """ GMM using Expectation-Maximization (EM) algorithm. """
    centroids, clusters = kmeans(X, k)  # Initialize GMM parameters using K-means
    means = centroids
    covariances = [np.cov(X[clusters == i].T) + np.eye(X.shape[1]) * 1e-6 for i in range(k)]
    weights = np.array([np.mean(clusters == i) for i in range(k)])
    
    log_likelihoods = []
    
    for iteration in range(max_iters):
        # E-step: calculate responsibilities
        responsibilities = expectation_step(X, means, covariances, weights)
        
        # M-step: update weights, means, and covariances
        means, covariances, weights = maximization_step(X, responsibilities)
        
        # Log-likelihood calculation
        log_likelihood = np.sum([
            np.log(np.sum([weights[i] * gaussian_pdf(X[j], means[i], covariances[i]) for i in range(k)]))
            for j in range(len(X))
        ])
        log_likelihoods.append(log_likelihood)
        
        # Check for convergence
        if len(log_likelihoods) > 1 and np.abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
            break
    
    return means, covariances, weights

# Step 3: Bayes Classifier using GMM

def classify_gmm(x, means, covariances, weights):
    """ Classify a point x using GMM. Return the class with the highest likelihood. """
    log_likelihoods = []
    for i in range(len(means)):
        log_likelihoods.append(weights[i] * gaussian_pdf(x, means[i], covariances[i]))
    return np.argmax(log_likelihoods)

def predict_gmm(X, gmms):
    """ Predict the class of each data point in X using trained GMMs for each class. """
    predictions = []
    for x in X:
        log_likelihoods = [np.sum([weights[i] * gaussian_pdf(x, mean, cov) 
                                   for i, (mean, cov, weight) in enumerate(gmm)]) for gmm in gmms]
        predictions.append(np.argmax(log_likelihoods))
    return predictions

# Step 4: Training and Classifying

def train_gmm_classifier(train_features_folder, k=5):
    """ Train GMM for each class. """
    gmm_models = {}
    label_dict = {'botanical_garden': 0, 'bus_interior': 1, 'elevator_shaft': 2}
    
    for class_name in label_dict:
        class_folder = os.path.join(train_features_folder, class_name)
        features, _ = load_features_from_folder(class_folder)
        
        # Initialize and train GMM using EM algorithm
        means, covariances, weights = gmm_em(features, k)
        gmm_models[label_dict[class_name]] = (means, covariances, weights)
    
    return gmm_models

def classify_test_images(test_features_folder, gmm_models):
    """ Classify test images using the trained GMM models. """
    correct = 0
    total = 0
    label_dict = {'botanical_garden': 0, 'bus_interior': 1, 'elevator_shaft': 2}
    
    for class_name in label_dict:
        class_folder = os.path.join(test_features_folder, class_name)
        features, _ = load_features_from_folder(class_folder)
        
        for feature in features:
            total += 1
            predicted_class = classify_gmm(feature, *gmm_models[label_dict[class_name]])
            if predicted_class == label_dict[class_name]:
                correct += 1
    
    accuracy = correct / total
    print(f"Accuracy: {accuracy * 100:.2f}%")
    return accuracy

# Main function to train and evaluate the model
def main():
    train_features_folder = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\train\output_train_feature'  # Folder where the train features are saved
    test_features_folder = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\test\output_test_feature'    # Folder where the test features are saved
    
    print("Training GMM models...")
    gmm_models = train_gmm_classifier(train_features_folder)
    
    print("Classifying test images...")
    classify_test_images(test_features_folder, gmm_models)

if __name__ == '__main__':
    main()


Training GMM models...


C:\Users\Abhishek Sharma\AppData\Local\Temp\ipykernel_15688\3844418337.py:65: RuntimeWarning: invalid value encountered in sqrt
  return np.exp(exponent) / np.sqrt((2 * np.pi)**n * cov_det)
E:\Anaconda\Lib\site-packages\numpy\linalg\linalg.py:2180: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

import os
import numpy as np

# Function to load feature vectors from a folder
def load_features_from_folder(folder_path):
    all_features = []
    all_labels = []
    label_dict = {'botanical_garden': 0, 'bus_interior': 1, 'elevator_shaft': 2}
    
    # Ensure the folder path exists
    if not os.path.exists(folder_path):
        raise ValueError(f"Directory {folder_path} does not exist.")
    
    # Iterate through all class folders and load feature vectors
    for class_folder in os.listdir(folder_path):
        class_folder_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_folder_path):
            for feature_file in os.listdir(class_folder_path):
                if feature_file.endswith(".npy"):  # Ensure we're reading only .npy files
                    feature_path = os.path.join(class_folder_path, feature_file)
                    features = np.load(feature_path)
                    all_features.append(features)
                    all_labels.extend([label_dict[class_folder]] * features.shape[0])
    
    if len(all_features) == 0:
        raise ValueError(f"No feature files found in {folder_path}. Please ensure that the feature extraction step has been completed successfully.")
    
    # Stack all features vertically to create a single feature array
    return np.vstack(all_features), np.array(all_labels)

# K-means clustering and GMM implementation from scratch remains the same

# Modified GMM to return log-likelihoods for plotting
def gmm_em(X, k, max_iters=100, tol=1e-6):
    centroids, clusters = kmeans(X, k)
    means = centroids
    covariances = [np.cov(X[clusters == i].T) + np.eye(X.shape[1]) * 1e-6 for i in range(k)]
    weights = np.array([np.mean(clusters == i) for i in range(k)])
    
    log_likelihoods = []
    
    for iteration in range(max_iters):
        responsibilities = expectation_step(X, means, covariances, weights)
        means, covariances, weights = maximization_step(X, responsibilities)
        
        # Log-likelihood calculation
        log_likelihood = np.sum([
            np.log(np.sum([weights[i] * gaussian_pdf(X[j], means[i], covariances[i]) for i in range(k)]))
            for j in range(len(X))
        ])
        log_likelihoods.append(log_likelihood)
        
        if len(log_likelihoods) > 1 and np.abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
            break
    
    return means, covariances, weights, log_likelihoods

# Train GMM with different cluster sizes and plot log-likelihood
def train_and_evaluate_gmm(train_features_folder, test_features_folder):
    num_clusters = [1, 2, 4, 6, 8, 16, 32, 64]
    label_dict = {'botanical_garden': 0, 'bus_interior': 1, 'elevator_shaft': 2}
    gmm_models_per_cluster = {}

    for k in num_clusters:
        print(f"\nTraining GMM models for k={k} clusters...")
        gmm_models = {}
        log_likelihoods_per_class = {}

        for class_name in label_dict:
            class_folder = os.path.join(train_features_folder, class_name)
            features, _ = load_features_from_folder(class_folder)
            
            # Initialize and train GMM using EM algorithm
            means, covariances, weights, log_likelihoods = gmm_em(features, k)
            gmm_models[label_dict[class_name]] = (means, covariances, weights)
            log_likelihoods_per_class[class_name] = log_likelihoods
        
        gmm_models_per_cluster[k] = gmm_models

        # Plotting log-likelihood vs iterations for each class
        plt.figure(figsize=(8, 6))
        for class_name, log_likelihoods in log_likelihoods_per_class.items():
            plt.plot(log_likelihoods, label=f'{class_name}')
        plt.title(f'Log-Likelihood vs Iterations for k={k}')
        plt.xlabel('Iterations')
        plt.ylabel('Log-Likelihood')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Evaluate performance for the current k value
        evaluate_gmm_on_test(test_features_folder, gmm_models, k)

def evaluate_gmm_on_test(test_features_folder, gmm_models, k):
    label_dict = {'botanical_garden': 0, 'bus_interior': 1, 'elevator_shaft': 2}
    y_true = []
    y_pred = []

    for class_name in label_dict:
        class_folder = os.path.join(test_features_folder, class_name)
        features, labels = load_features_from_folder(class_folder)
        y_true.extend(labels)
        
        for feature in features:
            predicted_class = classify_gmm(feature, *gmm_models[label_dict[class_name]])
            y_pred.append(predicted_class)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    accuracy = np.mean(y_true == y_pred)
    precision_per_class = precision_score(y_true, y_pred, average=None)
    recall_per_class = recall_score(y_true, y_pred, average=None)
    f1_per_class = f1_score(y_true, y_pred, average=None)

    mean_precision = np.mean(precision_per_class)
    mean_recall = np.mean(recall_per_class)
    mean_f1 = np.mean(f1_per_class)

    # Print metrics
    print(f'Performance for k={k} clusters:')
    print(f'Accuracy: {accuracy * 100:.2f}%')
    for i, class_name in enumerate(label_dict):
        print(f'Class {class_name}: Precision: {precision_per_class[i]:.4f}, Recall: {recall_per_class[i]:.4f}, F1-score: {f1_per_class[i]:.4f}')
    
    print(f'Mean Precision: {mean_precision:.4f}, Mean Recall: {mean_recall:.4f}, Mean F1-Score: {mean_f1:.4f}')

    # Confusion Matrix
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(f'Confusion Matrix (rows: actual, columns: predicted):\n{conf_matrix}')

# Main function to train and evaluate the model
def main():
    train_features_folder = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\train\output_train1_feature'  # Folder where the train features are saved
    test_features_folder  = r'E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\test\output_test1_feature'    # Folder where the test features are saved
    
    try:
        train_and_evaluate_gmm(train_features_folder, test_features_folder)
    except ValueError as e:
        print(f"Error: {e}")

if __name__ == '__main__':
    main()



Training GMM models for k=1 clusters...
Error: No feature files found in E:\SPR_Assignments\SPR_Assign02\group03\3class_scene_image_dataset\train\output_train1_feature\botanical_garden. Please ensure that the feature extraction step has been completed successfully.
